In [2]:
import pandas as pd

In [ ]:
import sys
sys.path.append('../game_on/')

from pln_model.limpieza import limpieza

In [5]:
import os
path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'raw_data', 'steam_games.csv')
df1 = pd.read_csv(path)

In [9]:
data_limpia = limpieza(df1)
data_limpia.columns

Index(['url', 'name', 'release_date', 'popular_tags', 'game_details',
       'languages', 'genre', 'game_description', 'original_price',
       'review_percentage', 'embedding'],
      dtype='object')

MODELO SBERT

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Configuración del Modelo
# Usamos un modelo balanceado entre velocidad y precisión
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

# 2. Simulación de Datos (Basado en el dataset de Steam de Kaggle)
# Cuando el equipo de datos te pase el CSV, cambiaremos esto por pd.read_csv()
data = {
    'game_id': [1, 2, 3, 4, 5],
    'title': ['Left 4 Dead 2', 'Stardew Valley', 'Elden Ring', 'Portal 2', 'Cyberpunk 2077'],
    'description': [
        'Zombies apocalypse cooperative shooter.',
        'Country life RPG with farming and animals.',
        'Challenging action RPG in a dark fantasy world.',
        'Physics-based puzzle game with portals.',
        'Open world action RPG set in a futuristic city.'
    ],
    'tags': ['Action, Zombies, Co-op', 'Farming, Simulation, Relaxing', 'Souls-like, Difficult, RPG', 'Puzzle, Sci-fi, Funny', 'Cyberpunk, RPG, Open World']
}

df = pd.DataFrame(data)

# 3. Creación del "Vibe Text" (Adaptación sugerida)
# SBERT funciona mejor si combinamos la descripción con los tags
df['metadata_combined'] = df['title'] + " " + df['description'] + " " + df['tags']

# 4. Generación de Embeddings
print(f"Vectorizando {len(df)} juegos...")
game_embeddings = model.encode(df['metadata_combined'].tolist(), convert_to_tensor=True)

# 5. Función de Recomendación Adaptada
def get_recommendations(user_query, n_top=3):
    # Vectorizar la consulta del usuario
    query_embedding = model.encode(user_query, convert_to_tensor=True)

    # Calcular similitud coseno
    cosine_scores = util.cos_sim(query_embedding, game_embeddings)[0]

    # Obtener los mejores N resultados
    top_results = torch.topk(cosine_scores, k=n_top)

    print(f"--- Recomendaciones para: '{user_query}' ---\n")
    for score, idx in zip(top_results.values, top_results.indices):
        game = df.iloc[idx.item()]
        print(f"🎯 Juego: {game['title']}")
        print(f"📊 Similitud: {score:.4f}")
        print(f"📝 Tags: {game['tags']}")
        print("-" * 40)

# PRUEBA DE FUNCIONAMIENTO
get_recommendations("I want a difficult game with magic and swords")

/Users/gonzaloneme/.pyenv/versions/3.10.6/envs/Game-On-Project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6071.01it/s]


Vectorizando 5 juegos...
--- Recomendaciones para: 'I want a difficult game with magic and swords' ---

🎯 Juego: Elden Ring
📊 Similitud: 0.5498
📝 Tags: Souls-like, Difficult, RPG
----------------------------------------
🎯 Juego: Stardew Valley
📊 Similitud: 0.3664
📝 Tags: Farming, Simulation, Relaxing
----------------------------------------
🎯 Juego: Cyberpunk 2077
📊 Similitud: 0.3271
📝 Tags: Cyberpunk, RPG, Open World
----------------------------------------


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Preparación de los Embeddings
# Si la columna 'embedding' ya tiene los vectores, los convertimos a un Tensor de PyTorch
# Si no están procesados, descomenta la línea de abajo para generarlos:
# game_embeddings = model.encode(data_limpia['game_description'].tolist(), convert_to_tensor=True)

# Asumiendo que 'embedding' ya existe y es una lista de listas o arrays:
game_embeddings = torch.tensor(data_limpia['embedding'].tolist())

# 3. Función de Recomendación Adaptada a tus columnas
def recomendador_game_on(query, n_top=5):
    # Vectorizamos la duda del usuario
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Cálculo de similitud coseno
    cosine_scores = util.cos_sim(query_embedding, game_embeddings)[0]

    # Buscamos los mejores N
    top_results = torch.topk(cosine_scores, k=n_top)

    print(f"🔎 Resultados para: '{query}'\n")
    print("="*50)

    for score, idx in zip(top_results.values, top_results.indices):
        game = data_limpia.iloc[idx.item()]

        print(f"🎮 JUEGO: {game['name']}")
        print(f"📊 Match: {score:.2%}")
        print(f"📂 Género: {game['genre']}")
        print(f"tags: {game['popular_tags']}")
        print(f"💰 Precio: {game['original_price']}")
        print(f"⭐ Reviews: {game['review_percentage']}")
        print(f"🔗 Link: {game['url']}")
        print("-" * 50)

# 4. Prueba del modelo con tu data real
recomendador_game_on("I want a strategy game about space and aliens")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5519.53it/s]


ValueError: too many dimensions 'str'